In [1]:
import zarr
import numpy as np
from viewer.utils import format_bytes

fs = 1000.0
duration = 3*60*60
n_channels = 32
n_samples = int(fs*duration)
chunk_samples = int(100*fs)
n_chunks = -(-n_samples//chunk_samples)

print(format_bytes(n_samples * n_channels * 8)) # float64
print(format_bytes(chunk_samples * n_channels))
print(n_chunks)

2.6 GiB
3.1 MiB
108


# Initialize zarr groups

In [2]:
root = zarr.open('exp1.zarr', mode='a')
root.tree()

/
├── behavior
│   └── pupil
│       ├── ts (648000,) float64
│       └── values (648000,) float64
├── ephys
│   ├── ts (10800000,) float64
│   └── values (10800000, 32) float64
└── units
    ├── spike_times (1571375,) float64
    └── spike_units (1571375,) uint8

## ephys data

In [4]:
ephys = root.require_group('ephys')

In [5]:
ephys.require_array(
    name='values',
    shape=(n_samples, n_channels),
    dtype=np.float64,
    chunks=(chunk_samples, n_channels),
)

<Array file://exp1.zarr/ephys/values shape=(10800000, 32) dtype=float64>

In [6]:
ephys['values'].info

Type               : Array
Zarr format        : 3
Data type          : Float64(endianness='little')
Fill value         : 0.0
Shape              : (10800000, 32)
Chunk shape        : (100000, 32)
Order              : C
Read-only          : False
Store type         : LocalStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 2764800000 (2.6G)

In [7]:
ephys.require_array(
    name='ts',
    shape=(n_samples),
    dtype=np.float64,
)

<Array file://exp1.zarr/ephys/ts shape=(10800000,) dtype=float64>

In [8]:
ephys['ts'].info

Type               : Array
Zarr format        : 3
Data type          : Float64(endianness='little')
Fill value         : 0.0
Shape              : (10800000,)
Chunk shape        : (168750,)
Order              : C
Read-only          : False
Store type         : LocalStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 86400000 (82.4M)

In [ ]:
ephys.attrs['fs'] = fs

In [60]:
list(ephys.attrs.items())

[('fs', 1000.0)]

In [ ]:
for cid in range(n_chunks):
    lo = cid*chunk_samples
    hi = min(cid+1, n_chunks-1)*chunk_samples
    ephys['values'][lo:hi] = np.random.normal(size=(chunk_samples, n_channels))

In [ ]:
ephys['ts'] = np.arange(0, duration, 1/fs)

## spikes data

In [63]:
import pynapple as nap

tsg = nap.load_file('./data/A5044-240404A_wake.nwb')['units']
tsg

  Index      rate    peak    SI    order
-------  --------  ------  ----  -------
      0  21.0089     2.49  0.85        4
      2  10.3802     0.18  1.13        0
      3   7.60317    3.43  1.02        7
      4   4.82503    2.91  0.3         6
      5   7.8838     1.81  1.34        3
      6   4.50536    3.64  0.73        9
      7   7.8329     2.54  1.73        5
      8  29.7255     5.37  2.26       11
      9  16.602      3.48  0.79        8
     23  24.7316     0.45  0.41        1
     24  14.338      0.86  0.64        2
     25   9.47692    6.2   0.62       12
     28  23.6827     5.21  0.4        10

In [65]:
units = root.require_group('units')
root.tree()

/
├── ephys
│   ├── ts (10800000,) float64
│   └── values (10800000, 32) float64
└── units

In [ ]:
def concat_tsg(tsg: nap.TsGroup) -> tuple[np.ndarray, np.ndarray]:
    spike_times = np.concatenate([tsg[k].t for k in tsg.keys()])
    spike_units = np.concatenate([np.full(len(tsg[k]), k) for k in tsg.keys()]).astype(np.uint8)

    sorted_idx = np.argsort(spike_times)
    spike_times = spike_times[sorted_idx]
    spike_units = spike_units[sorted_idx]

    return spike_times, spike_units

spike_times, spike_units = concat_tsg(tsg)
units.create_array("spike_times", data=spike_times)
units.create_array("spike_units", data=spike_units)

# metadata saved in JSON
metadata = tsg.metadata.to_dict()
for k, v in metadata.items():
    units.attrs[k] = v

In [153]:
root.tree()

/
├── ephys
│   ├── ts (10800000,) float64
│   └── values (10800000, 32) float64
└── units
    ├── spike_times (1571375,) float64
    └── spike_units (1571375,) uint8

## behavior

In [3]:
behavior = root.create_group('behavior')
pupil = behavior.create_group('pupil')
root.tree()

/
├── behavior
│   └── pupil
├── ephys
│   ├── ts (10800000,) float64
│   └── values (10800000, 32) float64
└── units
    ├── spike_times (1571375,) float64
    └── spike_units (1571375,) uint8

In [10]:
pupil.create_array(
    name='values',
    shape=(duration*60,),
    dtype=np.float64,
)

pupil.create_array(
    name='ts',
    shape=(duration*60,),
    dtype=np.float64,
)

<Array file://exp1.zarr/behavior/pupil/ts shape=(648000,) dtype=float64>

In [14]:
t = np.arange(0, duration, 1/60)
chord = np.sin(t)

pupil['values'][:] = chord
pupil['ts'][:] = t

In [16]:
pupil.attrs['fs'] = 60.0

In [17]:
root.tree()

/
├── behavior
│   └── pupil
│       ├── ts (648000,) float64
│       └── values (648000,) float64
├── ephys
│   ├── ts (10800000,) float64
│   └── values (10800000, 32) float64
└── units
    ├── spike_times (1571375,) float64
    └── spike_units (1571375,) uint8